<a href="https://colab.research.google.com/github/katerina-itopoulos/Where-You-LoRA-Matters/blob/main/experiments/finetuning_poc_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Hyperparameter Search

Environment Set Up

```
conda create -n fine_tune python=3.10 -y
conda activate fine_tune
conda install ipykernel -y
python -m ipykernel install --user --name fine_tune
pip install -r requirements.txt

````

And then:
```
gcloud auth application-default login
wandb login
```

In [1]:
!ls

sample_data


In [ ]:
!pip install -r /Applications/Documents/Where-You-LoRA-Matters/requirements.txt

ERROR: Could not open requirements file: [Errno 2] No such file or directory: '/Applications/Documents/Where-You-LoRA-Matters/requirements.txt'


In [1]:
!pip uninstall transformers -y
!pip install transformers==4.57.3

Found existing installation: transformers 4.57.3
Uninstalling transformers-4.57.3:
  Successfully uninstalled transformers-4.57.3
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 111.1 MB/s eta 0:00:00


In [ ]:
!pip install flash-attn --no-build-isolation

Imports

In [2]:
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from transformers import AutoProcessor, LlavaOnevisionForConditionalGeneration
import torch
import pandas as pd
from datasets import load_dataset
import random, json, math

from datasets import load_dataset
from PIL import Image
from transformers import AutoProcessor, AutoModelForCausalLM, BitsAndBytesConfig,AutoModelForImageTextToText
import gc
from datasets import load_dataset
from PIL import Image
import random

from typing import List, Dict
from PIL import Image
import torch, gc
import torch, gc
from typing import List, Tuple, Dict, Optional
from PIL import Image
import torch, random
from typing import List, Tuple, Dict, Optional
from PIL import Image
from datasets import Dataset
from transformers import AutoProcessor, AutoModelForImageTextToText
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch
from PIL import Image
from datasets import Image as HFImage
from datasets import load_dataset

gc.collect()
torch.cuda.empty_cache()

In [3]:
!pip install gcsfs

In [4]:
from google.colab import auth
auth.authenticate_user()

In [5]:
gc.collect()
torch.cuda.empty_cache()

In [9]:
for name in [
    "model","processor","outputs","out","H","V","inputs","IHS","V0","Vpr",
    "image_mask","text_mask","Vproj_vec","T_fused_vec","trainer"
]:
    if name in globals(): del globals()[name]

gc.collect()
torch.cuda.empty_cache()
print(torch.cuda.memory_summary())

KeyError: 'allocated_bytes.all.current'

In [ ]:
ds = load_dataset("merve/vqav2-small", split="validation")

In [ ]:
SEED = 42
REQ_TRAIN = 500
REQ_VALID = 100

def _ok(example):
    q = example.get("question") or ""
    return (q.strip() != "") and (example.get("image") is not None)

ds = ds.filter(_ok)

ds = ds.shuffle(seed=SEED)

N = len(ds)
n_train = min(REQ_TRAIN, max(0, N - REQ_VALID))
n_valid = min(REQ_VALID, N - n_train)


train_ds = ds.select(range(0, n_train))
valid_ds = ds.select(range(n_train, n_train + n_valid))

print(f"Total available: {N}")
print(f"Train: {len(train_ds)} | Valid: {len(valid_ds)}")


In [ ]:
print(train_ds[0])

In [ ]:
print(valid_ds[0])

Constants

In [10]:
SEED     = 42
BATCH    = 1
N_SAMPLES= 100
MAX_GEN  = 16

In [ ]:
idxs = list(range(len(ds))); random.Random(SEED).shuffle(idxs)
ds100 = ds.select(idxs[0:N_SAMPLES])
ds100 = ds100.cast_column("image", HFImage(decode=True))

Model set up

Qwen 3

In [ ]:
MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, trust_remote_code=True, torch_dtype=torch.bfloat16, device_map="auto"
).eval()

Metrics

In [11]:
import torch, json
import torch.nn.functional as F
EPS = 1e-8

def _to_cpu_f32(x):
    return None if x is None else x.detach().to(torch.float32, copy=False).cpu()

def _l2norm(X):  # [N,D]
    return X / (X.norm(dim=-1, keepdim=True).clamp_min(EPS))

def intra_modal_similarity(X):
    Xn = _l2norm(X); S = Xn @ Xn.T
    n = S.size(0)
    if n <= 1: return float("nan")
    mask = ~torch.eye(n, dtype=torch.bool)
    return S[mask].mean().item()

def intermodal_metrics(I, T):
    In, Tn = _l2norm(I), _l2norm(T)
    pos = (In * Tn).sum(-1)                            # matched pairs
    perm = torch.randperm(In.size(0))
    neg = (In * Tn[perm]).sum(-1)                      # mismatched
    margin = (pos.mean() - neg.mean()).item()
    auc = (pos[:,None] > neg[None,:]).float().mean().item() if pos.numel() and neg.numel() else float("nan")
    return {"pos_mean":pos.mean().item(), "neg_mean":neg.mean().item(),
            "margin":margin, "auc_proxy":auc}

def modality_gap(I, T, center=True):
    # I, T: [N, D] pooled image and text embeddings
    I = F.normalize(I, p=2, dim=-1)
    T = F.normalize(T, p=2, dim=-1)
    if center:
        I = I - I.mean(0, keepdim=True)
        T = T - T.mean(0, keepdim=True)
    mu_i = I.mean(0, keepdim=True)   # [1, D]
    mu_t = T.mean(0, keepdim=True)   # [1, D]
    # cosine distance between modality means (in [0, 2])
    gap = 1.0 - F.cosine_similarity(mu_i, mu_t).item()
    return gap

def effective_rank(Z, center=True):
    Z = Z.float()
    if center: Z = Z - Z.mean(0, keepdim=True)
    S = torch.linalg.svd(Z.to(torch.float64), full_matrices=False).S
    if S.numel() == 0: return float("nan")
    p = (S / S.sum().clamp_min(EPS)).clamp_min(EPS)
    H = -(p * p.log()).sum()
    return torch.exp(H).item()

def concentration_ratio(Z, k=1, center=True):
    Z = Z.float()
    if center: Z = Z - Z.mean(0, keepdim=True)
    S = torch.linalg.svd(Z.to(torch.float64), full_matrices=False).S
    if S.numel() == 0: return float("nan")
    k = min(k, S.numel())
    return (S[:k].sum() / S.sum().clamp_min(EPS)).item()

def linear_cka(X, Y, center=True):
    X = X.float(); Y = Y.float()
    if center:
        X = X - X.mean(0, keepdim=True)
        Y = Y - Y.mean(0, keepdim=True)
    K = X @ X.T; L = Y @ Y.T
    num = (K * L).sum()
    den = torch.sqrt((K*K).sum()) * torch.sqrt((L*L).sum())
    return (num / den).item() if den > EPS else float("nan")

def summarize_vectors(V0_all, Vproj_all, T_all):
    V0 = _to_cpu_f32(V0_all)
    V  = _to_cpu_f32(Vproj_all)
    T  = _to_cpu_f32(T_all)

    assert V is not None and T is not None, "Need Vproj_all and T_all"
    assert V.dim()==2 and T.dim()==2, f"Expect [N,D], got {tuple(V.shape)} & {tuple(T.shape)}"
    assert V.shape[0]==T.shape[0], "V/T batch sizes differ; ensure you pooled per-sample vectors"

    metrics = {
        "intra_sim_img": intra_modal_similarity(V),
        "intra_sim_txt": intra_modal_similarity(T),
        "modality_gap":  modality_gap(V, T),
        **intermodal_metrics(V, T),
        "erank_img":     effective_rank(V),
        "erank_txt":     effective_rank(T),
        "cr1_img":       concentration_ratio(V, k=1),
        "cr1_txt":       concentration_ratio(T, k=1),
        "cka_img_txt":   linear_cka(V, T),
    }
    if V0 is not None:
        metrics.update({
            "cka_V0_Vproj": linear_cka(V0, V),
            "intra_sim_V0": intra_modal_similarity(V0),
            "erank_V0":     effective_rank(V0),
            "cr1_V0":       concentration_ratio(V0, k=1),
        })
    return metrics


In [12]:
def _norm(x): return str(x).strip().lower()
def _gold_val(g):
    if g is None: return None
    return g[0]["answer"] if isinstance(g, list) and g and isinstance(g[0], dict) and "answer" in g[0] else g

def light_match(pred, gold):
    g = _gold_val(gold)
    return int(_norm(g) in _norm(pred)) if g is not None else 0

def delta_metrics(golds, full, text_only, image_only):
    N = len(golds)
    acc_full = sum(light_match(p,g) for p,g in zip(full,      golds)) / max(N,1)
    acc_txt  = sum(light_match(p,g) for p,g in zip(text_only, golds)) / max(N,1)
    acc_img  = sum(light_match(p,g) for p,g in zip(image_only,golds)) / max(N,1)
    return {
        "Acc_full": acc_full,
        "Acc_textOnly": acc_txt,
        "Acc_imageOnly": acc_img,
        "Delta_img": acc_full - acc_txt,
        "Delta_txt": acc_full - acc_img,
    }


Pipeline

In [13]:
def align_for_model(model, processor, inputs: dict) -> dict:
    import torch

    try:
        emb_dev = model.get_input_embeddings().weight.device
    except Exception:
        emb_dev = next(model.parameters()).device

    for k in ("input_ids", "attention_mask", "position_ids", "token_type_ids"):
        v = inputs.get(k, None)
        if isinstance(v, torch.Tensor):
            inputs[k] = v.to(emb_dev, non_blocking=True)

    visual_mod = getattr(model, "visual", model)
    first_param = next(visual_mod.parameters())
    vis_dev   = first_param.device
    vis_dtype = getattr(visual_mod, "dtype", first_param.dtype)

    pv = inputs.get("pixel_values", None)
    if isinstance(pv, torch.Tensor):
        inputs["pixel_values"] = pv.to(vis_dev, dtype=vis_dtype, non_blocking=True)

    grid = inputs.get("image_grid_thw", None)
    if isinstance(grid, torch.Tensor):
        inputs["image_grid_thw"] = grid.to(vis_dev, non_blocking=True)

    return inputs

In [14]:
def _coerce_visual_tokens(Vraw, B: int, D_llm: int):
    """
    Normalize various visual outputs to [B, T, D_llm].
    Handles:
      - tensor [B, T, D_llm]
      - tensor [T, D_llm] when B==1
      - tuple/list where the first tensor is [T, D_llm] or [B, T, D_llm]
    """
    import torch

    def _as_BTD(x):
        if not torch.is_tensor(x):
            return None
        if x.dim() == 3:
            if x.shape[0] == B and x.shape[-1] == D_llm:
                return x
            if x.shape[1] == B and x.shape[-1] == D_llm:
                return x.permute(1,0,2).contiguous()
        if x.dim() == 2 and x.shape[-1] == D_llm:
            if B == 1:
                return x.unsqueeze(0)
        return None

    cand = _as_BTD(Vraw)
    if cand is not None:
        return cand

    if isinstance(Vraw, (list, tuple)):
        for t in Vraw:
            cand = _as_BTD(t)
            if cand is not None:
                return cand

    return None

Conversations/data preprocessing

In [15]:
def make_conv_full(img: Image.Image, q: str) -> List[Dict]:
    return [{"role":"user","content":[{"type":"image","image":img},
                                      {"type":"text","text":q}]}]

def make_conv_text_only(q: str) -> List[Dict]:
    return [{"role":"user","content":[{"type":"text","text":q}]}]

def make_conv_image_only(img: Image.Image) -> List[Dict]:
    return [{"role":"user","content":[{"type":"image","image":img},
                                      {"type":"text","text":""}]}]

def build_convs_from_rows(
    rows: List[Dict],
    mode: str = "full",
) -> Tuple[List[List[Dict]], List[Image.Image], List[str]]:
    assert mode in {"full", "text_only", "image_only"}
    convs, imgs, qs = [], [], []
    for ex in rows:
        img, q = ex["image"], ex["question"]
        imgs.append(img); qs.append(q)

        if mode == "full":
            content = [{"type":"image","image":img}, {"type":"text","text":q}]
        elif mode == "text_only":
            content = [{"type":"text","text":q}]
        else:
            content = [{"type":"image","image":img}, {"type":"text","text":""}]

        convs.append([{"role":"user", "content": content}])

    return convs, imgs, qs

def build_convs_all_modes(rows):
    convs_full,  imgs, qs = build_convs_from_rows(rows, mode="full")
    convs_text,  _,   _   = build_convs_from_rows(rows, mode="text_only")
    convs_image, _,   _   = build_convs_from_rows(rows, mode="image_only")
    return convs_full, convs_text, convs_image, imgs, qs

In [16]:
def mean_pool_slice(x: torch.Tensor, start_idx: int = 0) -> torch.Tensor:
    if x.dim() == 2:
        x = x.unsqueeze(0)
    return x[:, start_idx:, :].mean(dim=1)

def forward_internals(model, processor, convs):
    inputs = processor.apply_chat_template(
        convs, add_generation_prompt=False, tokenize=True,
        return_tensors="pt", return_dict=True
    )
    inputs = align_for_model(model, processor, inputs)

    with torch.inference_mode():
        outs = model(
            **inputs,
            output_hidden_states=True,
            output_image_hidden_states=True,
            return_dict=True,
            use_cache=False,
        )
    H = outs.hidden_states[-1]
    B, _, D_llm = H.shape

    Vproj = None
    ihs = getattr(outs, "image_hidden_states", None)
    if ihs:
        for t in reversed(ihs):
            if isinstance(t, torch.Tensor) and t.dim() == 3 and t.shape[0] == B and t.shape[-1] == D_llm:
                Vproj = t
                break

    if Vproj is None:
        with torch.inference_mode():
            if "image_grid_thw" in inputs:
                Vraw = model.visual(inputs["pixel_values"], grid_thw=inputs["image_grid_thw"])
            else:
                Vraw = model.visual(inputs["pixel_values"])
        Vproj = _coerce_visual_tokens(Vraw, B, D_llm)  # -> [B, T_img, D_llm] or None

    return H, Vproj, inputs

def pool_vectors(V0: Optional[torch.Tensor], Vproj: torch.Tensor, H: torch.Tensor) -> Dict[str, torch.Tensor]:
    """
    V0:    [B, Ti0, D] or None
    Vproj: [B, Ti,  D]
    H:     [B, T_all, D] (fused LLM states with image tokens first)
    Returns CPU tensors: V0_vec, Vproj_vec, T_fused_vec (each [B, D])
    """
    out: Dict[str, torch.Tensor] = {}
    if V0 is not None:
        out["V0_vec"] = V0.mean(dim=1).to("cpu", non_blocking=True)

    out["Vproj_vec"] = Vproj.mean(dim=1).to("cpu", non_blocking=True)

    Ti = int(Vproj.shape[1]) if Vproj is not None else 0
    B, S = H.shape[:2]
    T_span = H[:, Ti:, :] if Ti < S else H  # fallback if no text tokens
    out["T_fused_vec"] = T_span.mean(dim=1).to("cpu", non_blocking=True)

    del H, Vproj, V0
    import gc, torch
    gc.collect(); torch.cuda.empty_cache()
    return out

def generate_answers(
    model,
    processor,
    imgs: List[Image.Image],
    qs: List[str],
    max_new_tokens: int = 16,
    do_sample: bool = False,
    temperature: float = 0.7,
    top_p: float = 0.8,
    top_k: int = 20,
    repetition_penalty: float = 1.0,
) -> List[str]:
    import torch, gc
    outs: List[str] = []

    pad_id = getattr(processor.tokenizer, "pad_token_id", None)
    if pad_id is None:
        pad_id = processor.tokenizer.eos_token_id

    for img, q in zip(imgs, qs):
        conv = [{"role": "user", "content": [
            {"type": "image", "image": img},
            {"type": "text",  "text": q},
        ]}]
        gen_inp = processor.apply_chat_template(
            conv, add_generation_prompt=True, tokenize=True,
            return_tensors="pt", return_dict=True
        )
        gen_inp = align_for_model(model, processor, gen_inp)

        with torch.inference_mode():
            gen_ids = model.generate(
                **gen_inp,
                max_new_tokens=max_new_tokens,
                do_sample=do_sample,
                temperature=temperature,
                top_p=top_p,
                top_k=top_k,
                repetition_penalty=repetition_penalty,
                eos_token_id=processor.tokenizer.eos_token_id,
                pad_token_id=pad_id,
            )
        start = gen_inp["input_ids"].shape[1]
        outs.append(processor.tokenizer.decode(gen_ids[0][start:], skip_special_tokens=True).strip())

        del gen_ids, gen_inp
        gc.collect(); torch.cuda.empty_cache()
    return outs

def generate_batch(
    model,
    processor,
    convs: List[List[Dict]],
    max_new_tokens: int = 50,
    do_sample: bool = False,
    temperature: float = 0.0,
    top_p: float = 1.0,
    top_k: int = 1,
    repetition_penalty: float = 1.0,
) -> List[str]:
    import torch, gc
    outs: List[str] = []

    pad_id = getattr(processor.tokenizer, "pad_token_id", None)
    if pad_id is None:
        pad_id = processor.tokenizer.eos_token_id

    for conv in convs:
        gen_inp = processor.apply_chat_template(
            conv, add_generation_prompt=True, tokenize=True,
            return_tensors="pt", return_dict=True
        )
        gen_inp = align_for_model(model, processor, gen_inp)

        with torch.inference_mode():
            ids = model.generate(
                **gen_inp,
                max_new_tokens=max_new_tokens,
                do_sample=do_sample,
                temperature=temperature,
                top_p=top_p,
                top_k=top_k,
                repetition_penalty=repetition_penalty,
                eos_token_id=processor.tokenizer.eos_token_id,
                pad_token_id=pad_id,
            )
        start = gen_inp["input_ids"].shape[1]
        outs.append(processor.tokenizer.decode(ids[0][start:], skip_special_tokens=True).strip())

        del ids, gen_inp
        gc.collect(); torch.cuda.empty_cache()
    return outs


In [17]:
def run_batch_block_vectors(
    model, processor, ds, start: int, end: int,
    do_sample: bool = False,
    temperature: float = 0.0,
    top_p: float = 1.0,
    top_k: int = 1,
    repetition_penalty: float = 1.0,
    max_new_tokens: int = 50,
):
    import torch, gc

    batch = ds.select(list(range(start, end)))
    rows  = [batch[i] for i in range(len(batch))]
    if not rows:
        return [], [], [], []

    convs_full, imgs, qs = build_convs_from_rows(rows, mode="full")

    #tokenize multimodal chats
    inputs = processor.apply_chat_template(
        convs_full, add_generation_prompt=False, tokenize=True,
        return_tensors="pt", return_dict=True
    )
    inputs = align_for_model(model, processor, inputs)

    #fused LLM states (last layer)
    with torch.inference_mode():
        outs = model(**inputs, output_hidden_states=True, return_dict=True, use_cache=False)
        H = outs.hidden_states[-1]

    B, _, D_llm = H.shape
    with torch.inference_mode():
        Vraw = (model.visual(inputs["pixel_values"], grid_thw=inputs["image_grid_thw"])
                if "image_grid_thw" in inputs else
                model.visual(inputs["pixel_values"]))

    Vproj = _coerce_visual_tokens(Vraw, B, D_llm)

    if Vproj is None:
        print(f"[warn] skip block {start}:{end} (cannot coerce visual to [B,T,{D_llm}])")
        return [], [], [], []

    #pool to per-sample vectors on CPU
    Ti = int(Vproj.shape[1])                                  # image token count inserted first
    Vproj_vec   = Vproj.mean(1).to("cpu", non_blocking=True)  # [B, D_llm]
    T_span      = H[:, Ti:, :] if Ti < H.shape[1] else H
    T_fused_vec = T_span.mean(1).to("cpu", non_blocking=True) # [B, D_llm]
    V0_vec_list = []  # no pre-proj on Qwen path

    assert Vproj_vec.shape[0] == T_fused_vec.shape[0], "batch mismatch after pooling"

    #free GPU before generation
    del outs, H, Vproj, inputs
    gc.collect(); torch.cuda.empty_cache()

    #GENERATION (FULL only)
    answers_full = generate_answers(
        model, processor, imgs, qs,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        repetition_penalty=repetition_penalty,
    )

    # return lists (each entry is a [B, D] tensor on CPU)
    return V0_vec_list, [Vproj_vec], [T_fused_vec], answers_full

In [18]:
def run_pipeline_vectors(model, processor, ds, batch_size: int = 1):
    import gc, torch
    all_V0, all_Vproj, all_T = [], [], []
    for s in range(0, len(ds), batch_size):
        e = min(s + batch_size, len(ds))
        V0_l, Vp_l, T_l, _ = run_batch_block_vectors(model, processor, ds, s, e)
        if V0_l: all_V0.extend(V0_l)
        all_Vproj.extend(Vp_l); all_T.extend(T_l)
        del V0_l, Vp_l, T_l; gc.collect(); torch.cuda.empty_cache()
    V0_all    = torch.cat(all_V0,    0) if all_V0 else None
    Vproj_all = torch.cat(all_Vproj, 0)
    T_all     = torch.cat(all_T,     0)
    assert Vproj_all.shape[0] == T_all.shape[0]
    return V0_all, Vproj_all, T_all

def run_pipeline_100(
    model, processor, ds100, batch_size: int = 1,
    do_sample: bool = False,
    temperature: float = 0.0,
    top_p: float = 1.0,
    top_k: int = 1,
    repetition_penalty: float = 1.0,
    max_new_tokens: int = 50,
):
    import gc, torch
    all_V0, all_Vproj, all_T, answers = [], [], [], []

    for s in range(0, len(ds100), batch_size):
        e = min(s + batch_size, len(ds100))
        V0_l, Vp_l, T_l, ans_full = run_batch_block_vectors(
            model, processor, ds100, s, e,
            do_sample=do_sample,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            repetition_penalty=repetition_penalty,
            max_new_tokens=max_new_tokens,
        )
        if V0_l: all_V0.extend(V0_l)
        all_Vproj.extend(Vp_l); all_T.extend(T_l); answers.extend(ans_full)
        del V0_l, Vp_l, T_l, ans_full; gc.collect(); torch.cuda.empty_cache()

    V0_all    = torch.cat(all_V0,    0) if all_V0 else None
    Vproj_all = torch.cat(all_Vproj, 0)
    T_all     = torch.cat(all_T,     0)
    assert Vproj_all.dim()==2 and T_all.dim()==2
    assert Vproj_all.shape[0] == T_all.shape[0]
    return V0_all, Vproj_all, T_all, answers

In [19]:
def run_pipeline_100_masked(
    model,
    processor,
    ds100,
    batch_size: int = 1,
    max_new_tokens: int = 16,
    do_sample: bool = False,
    temperature: float = 0.0,
    top_p: float = 1.0,
    top_k: int = 1,
    repetition_penalty: float = 1.0,
):
    """
    Functional probe for modality collapse:
      - FULL, TEXT-ONLY, IMAGE-ONLY generations per batch
      - Returns predictions, golds, and Δ metrics (Δ_img, Δ_txt)
    """
    import gc, torch
    ans_full, ans_text, ans_image, golds = [], [], [], []

    for s in range(0, len(ds100), batch_size):
        e = min(s + batch_size, len(ds100))
        rows = [ds100[i] for i in range(s, e)]

        # Build message lists (each item is ONE message dict)
        convs_full_msgs,  _, _ = build_convs_from_rows(rows, mode="full")
        convs_text_msgs,  _, _ = build_convs_from_rows(rows, mode="text_only")
        convs_image_msgs, _, _ = build_convs_from_rows(rows, mode="image_only")

        # Wrap each single-message dict into a conversation (list of messages)
        convs_full  = [[m] for m in convs_full_msgs]
        convs_text  = [[m] for m in convs_text_msgs]
        convs_image = [[m] for m in convs_image_msgs]

        # Batched generation per mode
        full_preds  = generate_batch(
            model, processor, convs_full,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample, temperature=temperature,
            top_p=top_p, top_k=top_k,
            repetition_penalty=repetition_penalty,
        )
        text_preds  = generate_batch(
            model, processor, convs_text,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample, temperature=temperature,
            top_p=top_p, top_k=top_k,
            repetition_penalty=repetition_penalty,
        )
        image_preds = generate_batch(
            model, processor, convs_image,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample, temperature=temperature,
            top_p=top_p, top_k=top_k,
            repetition_penalty=repetition_penalty,
        )

        ans_full.extend(full_preds)
        ans_text.extend(text_preds)
        ans_image.extend(image_preds)

        # Golds: VQAv2-style list of dicts or string (MC datasets)
        for ex in rows:
            golds.append(ex.get("answers") or ex.get("multiple_choice_answer"))

        del (rows, convs_full_msgs, convs_text_msgs, convs_image_msgs,
             convs_full, convs_text, convs_image,
             full_preds, text_preds, image_preds)
        gc.collect(); torch.cuda.empty_cache()

    deltas = delta_metrics(golds, ans_full, ans_text, ans_image)
    answers_pack = {"full": ans_full, "text_only": ans_text, "image_only": ans_image}
    return answers_pack, golds, deltas


Run Pipeline

In [ ]:
V0_all, Vproj_all, T_all, answers = run_pipeline_100(
    model,
    processor,
    ds100,
    batch_size=1,
    max_new_tokens=50,
    do_sample=False,           # greedy decoding for deterministic output
    temperature=0.0,           # disables randomness
    top_p=1.0,                 # include all tokens (no nucleus sampling)
    top_k=1,                   # pick single most probable token (greedy)
    repetition_penalty=1.0
)

In [ ]:
print("shapes:",
      None if V0_all is None else V0_all.shape,
      Vproj_all.shape, T_all.shape, len(answers))

In [ ]:
metrics = summarize_vectors(V0_all, Vproj_all, T_all)
print(json.dumps(metrics, indent=2))

In [ ]:
answers_pack, golds, deltas = run_pipeline_100_masked(model, processor, ds100, batch_size=1, max_new_tokens=16)
print("Δ metrics:", deltas)

We want to log effective rank and concentration ratio during finetuning too on a small portion of the validation set


Fine Tuning example run

- lets finetune qwen 3, due to its Lora compatibility
- we can do a trial on lets say a 5k dataset with low rank/resources to double check everything

In [8]:
#!pip uninstall -y transformers tokenizers huggingface-hub accelerate peft
!pip install "accelerate>=0.30" "tokenizers>=0.19" "safetensors" "peft>=0.11.0" qwen_vl_utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 18.6 MB/s eta 0:00:00


In [20]:
!wandb login

wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Find your API key here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter: 
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: kattyboomboom (katerina-itopoulos-srh-berlin) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [21]:
!rm -rf ~/.cache/huggingface/*
!df -h

Filesystem      Size  Used Avail Use% Mounted on
overlay         108G   39G   70G  36% /
tmpfs            64M     0   64M   0% /dev
shm             5.8G     0  5.8G   0% /dev/shm
/dev/root       2.0G  1.2G  750M  62% /usr/sbin/docker-init
tmpfs           6.4G  896K  6.4G   1% /var/colab
/dev/sda1        73G   40G   34G  55% /kaggle/input
tmpfs           6.4G     0  6.4G   0% /proc/acpi
tmpfs           6.4G     0  6.4G   0% /proc/scsi
tmpfs           6.4G     0  6.4G   0% /sys/firmware


In [22]:
import torch
import numpy as np
from transformers import (
    Qwen3VLForConditionalGeneration,
    AutoProcessor,
    TrainingArguments,
    TrainerCallback,
    EarlyStoppingCallback,
    Trainer
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from datasets import load_dataset
from qwen_vl_utils import process_vision_info
import wandb
import os
from datetime import datetime

In [ ]:
#https://huggingface.co/docs/optimum-neuron/en/training_tutorials/finetune_qwen3

Dataset

In [23]:
def cleanup_hf_cache():
    """Clean up HuggingFace cache to free disk space"""
    import shutil
    from pathlib import Path

    # HuggingFace cache locations
    cache_dirs = [
        Path.home() / ".cache" / "huggingface" / "datasets",
        Path.home() / ".cache" / "huggingface" / "hub",
    ]

    print("\n🧹 Cleaning up HuggingFace cache...")
    total_freed = 0

    for cache_dir in cache_dirs:
        if cache_dir.exists():
            size_before = sum(f.stat().st_size for f in cache_dir.rglob('*') if f.is_file())
            size_before_mb = size_before / (1024 * 1024)

            # Remove cache directory
            shutil.rmtree(cache_dir, ignore_errors=True)
            cache_dir.mkdir(parents=True, exist_ok=True)

            total_freed += size_before_mb
            print(f"  Cleared {cache_dir.name}: {size_before_mb:.1f} MB")

    print(f"✓ Total space freed: {total_freed:.1f} MB\n")

In [24]:
cleanup_hf_cache()


🧹 Cleaning up HuggingFace cache...
✓ Total space freed: 0.0 MB



In [25]:
def prepare_vqav2_datasets_preprocessed(
    processor,
    train_size=1000,
    val_size=200,
    test_size=200,
    cache_dir=None,
    gcs_bucket="where_you_lora_matters_thesis",
    gcs_prefix="datasets/preprocessed_vqa"
):
    """
    Prepare and preprocess VQAv2 datasets with GCS caching
    Uses your existing GCS bucket: where_you_lora_matters_thesis/datasets
    """
    from datasets import load_dataset, Dataset
    from qwen_vl_utils import process_vision_info
    from itertools import islice
    import gcsfs

    # GCS paths
    gcs_train_path = f"gs://{gcs_bucket}/{gcs_prefix}/train_{train_size}"
    gcs_val_path = f"gs://{gcs_bucket}/{gcs_prefix}/val_{val_size}"
    gcs_test_path = f"gs://{gcs_bucket}/{gcs_prefix}/test_{test_size}"

    # Check if already preprocessed in GCS
    try:
        print("🔍 Checking for preprocessed data in GCS...")
        fs = gcsfs.GCSFileSystem()

        train_exists = fs.exists(f"{gcs_bucket}/{gcs_prefix}/train_{train_size}")
        val_exists = fs.exists(f"{gcs_bucket}/{gcs_prefix}/val_{val_size}")
        test_exists = fs.exists(f"{gcs_bucket}/{gcs_prefix}/test_{test_size}")

        if train_exists and val_exists and test_exists:
            print("✓ Loading preprocessed datasets from GCS...")
            train_dataset = Dataset.load_from_disk(gcs_train_path)
            val_dataset = Dataset.load_from_disk(gcs_val_path)
            test_dataset = Dataset.load_from_disk(gcs_test_path)

            print(f"✓ Loaded from GCS: {len(train_dataset)} train, {len(val_dataset)} val, {len(test_dataset)} test")
            return train_dataset, val_dataset, test_dataset
    except Exception as e:
        print(f"   No cached data found in GCS (will preprocess): {e}")

    # Preprocess from scratch
    print(f"\n📊 Preprocessing VQAv2 dataset (first time only)...")

    train_dataset_stream = load_dataset("lmms-lab/VQAv2", split="validation", streaming=True, cache_dir=cache_dir)
    testdev_dataset_stream = load_dataset("lmms-lab/VQAv2", split="testdev", streaming=True, cache_dir=cache_dir)

    train_examples = list(islice(train_dataset_stream, train_size))
    val_examples = list(islice(testdev_dataset_stream, val_size))
    test_examples = list(islice(testdev_dataset_stream.skip(val_size), test_size))

    print(f"✓ Downloaded {len(train_examples) + len(val_examples) + len(test_examples)} samples")

    def format_and_preprocess(example):
        """Fully preprocess example"""
        if isinstance(example.get('answer'), list):
            answer = example['answer'][0] if example['answer'] else ""
        else:
            answer = example.get('answer', example.get('multiple_choice_answer', ''))

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": example['image']},
                    {"type": "text", "text": example['question']}
                ]
            },
            {
                "role": "assistant",
                "content": [{"type": "text", "text": str(answer)}]
            }
        ]

        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        image_inputs, video_inputs = process_vision_info(messages)

        inputs = processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        )

        result = {
            "input_ids": inputs['input_ids'][0].tolist(),
            "attention_mask": inputs['attention_mask'][0].tolist(),
        }

        if 'pixel_values' in inputs and inputs['pixel_values'] is not None:
            pv = inputs['pixel_values']
            if pv.dim() == 5:
                pv = pv[0]
            result['pixel_values'] = pv.tolist()

        if 'image_grid_thw' in inputs and inputs['image_grid_thw'] is not None:
            igt = inputs['image_grid_thw']
            if igt.dim() == 2:
                igt = igt[0]
            result['image_grid_thw'] = igt.tolist()

        return result

    print("Preprocessing train set...")
    train_formatted = [format_and_preprocess(ex) for ex in train_examples]

    print("Preprocessing val set...")
    val_formatted = [format_and_preprocess(ex) for ex in val_examples]

    print("Preprocessing test set...")
    test_formatted = [format_and_preprocess(ex) for ex in test_examples]

    train_dataset = Dataset.from_list(train_formatted)
    val_dataset = Dataset.from_list(val_formatted)
    test_dataset = Dataset.from_list(test_formatted)

    # Save to GCS
    print(f"\n💾 Saving to GCS: gs://{gcs_bucket}/{gcs_prefix}/...")
    try:
        train_dataset.save_to_disk(gcs_train_path)
        val_dataset.save_to_disk(gcs_val_path)
        test_dataset.save_to_disk(gcs_test_path)
        print(f"✓ Saved to GCS! All future runs will load instantly.")
    except Exception as e:
        print(f"⚠️  Warning: Could not save to GCS: {e}")
        print(f"   Datasets still ready to use in memory.")

    return train_dataset, val_dataset, test_dataset

Metrics

In [26]:
class WandBLoRAMetricsCallback(TrainerCallback):
    """Logs LoRA-specific metrics to W&B"""

    def __init__(self, compute_freq=5):
        self.compute_freq = compute_freq
        self.best_val_loss = float('inf')

    def on_log(self, args, state, control, model, logs=None, **kwargs):
        """Log metrics to W&B at each logging step"""
        if logs is None:
            return

        # Compute LoRA metrics periodically
        if state.global_step % self.compute_freq == 0:
            lora_metrics = self.compute_lora_metrics(model)
            wandb.log(lora_metrics, step=state.global_step)
            self._print_lora_metrics(lora_metrics, state.global_step)

        # Track best model
        if 'eval_loss' in logs:
            if logs['eval_loss'] < self.best_val_loss:
                self.best_val_loss = logs['eval_loss']
                wandb.run.summary["best_val_loss"] = self.best_val_loss
                wandb.run.summary["best_step"] = state.global_step

    def compute_lora_metrics(self, model):
        """Compute LoRA-specific metrics"""
        all_eranks = []
        all_stable_ranks = []
        all_frobenius_norms = []
        all_spectral_norms = []

        for name, module in model.named_modules():
            if hasattr(module, 'lora_A') and hasattr(module, 'lora_B'):
                with torch.no_grad():
                    lora_A = module.lora_A['default'].weight.float()
                    lora_B = module.lora_B['default'].weight.float()
                    lora_weight = lora_B @ lora_A

                    # Compute singular values
                    s = torch.linalg.svdvals(lora_weight)

                    # Effective Rank
                    s_normalized = s / (s.sum() + 1e-10)
                    entropy = -(s_normalized * torch.log(s_normalized + 1e-10)).sum()
                    erank = torch.exp(entropy).item()

                    # Stable Rank
                    frobenius_norm = torch.norm(lora_weight, p='fro').item()
                    spectral_norm = s[0].item()
                    stable_rank = (frobenius_norm ** 2) / (spectral_norm ** 2 + 1e-10)

                    all_eranks.append(erank)
                    all_stable_ranks.append(stable_rank)
                    all_frobenius_norms.append(frobenius_norm)
                    all_spectral_norms.append(spectral_norm)

        metrics = {}
        if all_eranks:
            metrics['lora/effective_rank_mean'] = np.mean(all_eranks)
            metrics['lora/effective_rank_std'] = np.std(all_eranks)
            metrics['lora/stable_rank_mean'] = np.mean(all_stable_ranks)
            metrics['lora/frobenius_norm_mean'] = np.mean(all_frobenius_norms)

            # Rank utilization
            nominal_rank = lora_A.shape[0]
            metrics['lora/rank_utilization'] = np.mean(all_eranks) / nominal_rank

        return metrics

    def _print_lora_metrics(self, metrics, step):
        """Print LoRA metrics summary"""
        print(f"\n{'='*70}")
        print(f"LoRA Metrics at Step {step}")
        print(f"{'='*70}")
        print(f"Effective Rank: {metrics.get('lora/effective_rank_mean', 0):.2f} ± {metrics.get('lora/effective_rank_std', 0):.2f}")
        print(f"Rank Utilization: {metrics.get('lora/rank_utilization', 0)*100:.1f}%")
        print(f"{'='*70}\n")

Model Set Up

In [27]:
def setup_vl_model_and_processor(model_name, lora_config, min_pixels=128*32*32, max_pixels=256*32*32):
    """
    Load vision-language model and apply LoRA for Qwen3-VL

    Args:
        model_name: HuggingFace model name (e.g., "Qwen/Qwen3-VL-8B-Instruct")
        lora_config: LoraConfig object
        min_pixels: Min image resolution (Qwen3-VL uses 32x32 patches!)
        max_pixels: Max image resolution

    Returns:
        model, processor, trainable_params, total_params
    """
    from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
    from peft import get_peft_model
    import torch

    print(f"\n📦 Loading vision-language model...")
    print(f"   Model: {model_name}")

    # Load processor
    processor = AutoProcessor.from_pretrained(
        model_name,
        min_pixels=min_pixels,
        max_pixels=max_pixels,
        trust_remote_code=True
    )

    # Load Qwen3-VL model with Flash Attention 2
    print("   Loading Qwen3-VL model...")
    model = Qwen3VLForConditionalGeneration.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
        attn_implementation="sdpa",  # Enable Flash Attention 2
    )
    print(f"   ✓ Loaded: {type(model).__name__}")

    # CRITICAL: Freeze vision encoder BEFORE applying LoRA
    print("   🔒 Freezing vision encoder...")
    if hasattr(model, 'visual'):
        for param in model.visual.parameters():
            param.requires_grad = False
        print("   ✓ Vision encoder frozen")

    model.train()

    print(f"\n⚙️  Applying LoRA (rank={lora_config.r}, alpha={lora_config.lora_alpha})...")
    print(f"   Target modules: {lora_config.target_modules}")

    # Apply LoRA (vision already frozen)
    model = get_peft_model(model, lora_config)
    print("   ✓ LoRA applied")

    # Enable gradient checkpointing
    model.gradient_checkpointing_enable()

    if hasattr(model, 'enable_input_require_grads'):
        model.enable_input_require_grads()

    # Count parameters
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    trainable_pct = 100 * trainable_params / total_params

    print(f"✓ Trainable params: {trainable_params:,} ({trainable_pct:.2f}%)")

    # Verify
    if trainable_params == 0:
        raise RuntimeError("No trainable parameters!")

    return model, processor, trainable_params, total_params

LoRA

In [28]:
def create_lora_config_vl(lora_r, lora_alpha, lora_dropout, target_modules):
    """
    Create LoRA configuration for Qwen2.5-VL

    Target modules for Qwen2.5-VL language model part:
    - q_proj, k_proj, v_proj, o_proj (attention)
    - gate_proj, up_proj, down_proj (FFN)

    Note: We typically don't apply LoRA to the vision encoder
    """
    return LoraConfig(
        r=lora_r,
        lora_alpha=lora_alpha,
        lora_dropout=lora_dropout,
        target_modules=target_modules,
        bias="none",
        inference_mode=False,
        task_type=TaskType.CAUSAL_LM,
    )

Data collator

In [29]:
class VLDataCollator:
    """Custom data collator for vision-language models"""

    def __init__(self, processor):
        self.processor = processor

    def __call__(self, features):
        """Collate batch of vision-language examples"""
        # Extract text fields
        input_ids = [f['input_ids'] for f in features]
        attention_mask = [f['attention_mask'] for f in features]

        # Pad text sequences
        batch = self.processor.tokenizer.pad(
            {"input_ids": input_ids, "attention_mask": attention_mask},
            padding=True,
            return_tensors="pt"
        )

        # Ensure correct dtypes for text
        batch['input_ids'] = batch['input_ids'].long()
        batch['attention_mask'] = batch['attention_mask'].long()

        # Handle vision inputs - concatenate all tiles from all images
        pixel_values_list = []
        image_grid_thw_list = []

        for f in features:
            if 'pixel_values' in f and f['pixel_values'] is not None:
                pv = f['pixel_values']
                igt = f['image_grid_thw']

                # Ensure they're tensors
                if not torch.is_tensor(pv):
                    pv = torch.tensor(pv)
                if not torch.is_tensor(igt):
                    igt = torch.tensor(igt)

                pixel_values_list.append(pv)  # Each is (num_tiles, C, H, W)
                image_grid_thw_list.append(igt)  # Each is (3,)

        # Concatenate vision inputs if present
        if pixel_values_list:
            # Concatenate all tiles from all images: (total_tiles, C, H, W)
            batch['pixel_values'] = torch.cat(pixel_values_list, dim=0)
            # Stack grid info: (batch_size, 3)
            batch['image_grid_thw'] = torch.stack(image_grid_thw_list, dim=0)

        # Labels for language modeling
        batch['labels'] = batch['input_ids'].clone().long()

        return batch

Training code

In [30]:
def setup_optimizer_scheduler(
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_steps=5,
    weight_decay=0.01,
):
    """
    Configure optimizer and learning rate scheduler

    Args:
        learning_rate: Peak learning rate
        lr_scheduler_type: Type of LR scheduler (cosine, linear, constant, etc.)
        warmup_steps: Number of warmup steps
        weight_decay: Weight decay coefficient

    Returns:
        dict: Configuration for optimizer and scheduler
    """
    return {
        "learning_rate": learning_rate,
        "lr_scheduler_type": lr_scheduler_type,
        "warmup_steps": warmup_steps,
        "weight_decay": weight_decay,
    }

In [31]:
def initialize_wandb(project_name, run_name, entity, config_dict):
    """Initialize W&B run"""
    print(f"\n{'='*70}")
    print("🚀 Initializing Weights & Biases")
    print(f"{'='*70}")

    wandb.init(
        project=project_name,
        name=run_name,
        entity=entity,
        config=config_dict
    )

    print(f"✓ W&B Run: {wandb.run.name}")
    print(f"✓ URL: {wandb.run.url}")

In [32]:
def upload_to_wandb_artifacts(
    output_dir, run_name, lora_r, lora_alpha,
    model_name, best_val_loss, train_size, val_size
):
    """Upload LoRA checkpoint to W&B Artifacts"""
    print("\n📦 Uploading LoRA checkpoint to W&B Artifacts...")

    artifact = wandb.Artifact(
        name=f"lora-checkpoint-{run_name}",
        type="model",
        description=f"LoRA adapter (rank={lora_r}) for {model_name}",
        metadata={
            "lora_r": lora_r,
            "lora_alpha": lora_alpha,
            "base_model": model_name,
            "best_val_loss": best_val_loss,
            "train_size": train_size,
            "val_size": val_size,
        }
    )

    artifact.add_dir(output_dir)
    wandb.log_artifact(artifact)

    print(f"✓ LoRA checkpoint uploaded to W&B!")
    return artifact

In [33]:
def train_vl_lora_with_wandb(
    # Required inputs
    model,
    train_dataset,
    val_dataset,
    test_dataset,
    processor,

    # W&B config
    wandb_project="lora-vqav2",
    wandb_run_name=None,
    wandb_entity=None,
    wandb_config=None,  # Additional config to log to W&B

    # Output
    output_dir="./lora-vqav2-output",

    # Training schedule
    epochs=3,
    max_steps=50,
    eval_steps=10,
    logging_steps=5,
    save_steps=None,  # Default: same as eval_steps

    # Batch size
    batch_size=1,
    gradient_accumulation_steps=4,

    # Optimization (pass dict from setup_optimizer_scheduler)
    optimizer_config=None,

    # Early stopping
    early_stopping_patience=3,
    early_stopping_threshold=0.01,
):
    """
    Main training function - pure orchestration of training loop

    Args:
        model: PEFT model with LoRA already applied
        train_dataset: Prepared training dataset
        val_dataset: Prepared validation dataset
        test_dataset: Prepared test dataset
        processor: Processor for collation
        wandb_project: W&B project name
        wandb_run_name: W&B run name
        wandb_entity: W&B entity (username/team)
        wandb_config: Additional config dict to log to W&B
        output_dir: Where to save checkpoints
        max_steps: Maximum training steps
        eval_steps: Evaluate every N steps
        logging_steps: Log every N steps
        save_steps: Save checkpoint every N steps (default: same as eval_steps)
        batch_size: Per-device batch size
        gradient_accumulation_steps: Gradient accumulation steps
        optimizer_config: Dict from setup_optimizer_scheduler()
        early_stopping_patience: Patience for early stopping
        early_stopping_threshold: Threshold for early stopping

    Returns:
        trainer: Trained Trainer object
    """

    # Set defaults
    if save_steps is None:
        save_steps = eval_steps

    if optimizer_config is None:
        optimizer_config = setup_optimizer_scheduler()

    if wandb_config is None:
        wandb_config = {}

    # 1. Initialize W&B
    print(f"\n{'='*70}")
    print("🚀 Initializing Weights & Biases")
    print(f"{'='*70}")

    # Build W&B config
    full_config = {
        "train_size": len(train_dataset),
        "val_size": len(val_dataset),
        "test_size": len(test_dataset),
        "trainable_params": sum(p.numel() for p in model.parameters() if p.requires_grad),
        "total_params": sum(p.numel() for p in model.parameters()),
        "max_steps": max_steps,
        "batch_size": batch_size,
        "gradient_accumulation_steps": gradient_accumulation_steps,
        **optimizer_config,
        **wandb_config,  # User-provided config
    }
    full_config["trainable_percent"] = 100 * full_config["trainable_params"] / full_config["total_params"]

    wandb.init(
        project=wandb_project,
        name=wandb_run_name,
        entity=wandb_entity,
        config=full_config
    )

    print(f"✓ W&B Run: {wandb.run.name}")
    print(f"✓ URL: {wandb.run.url}")

    # 2. Create training arguments
    training_args = TrainingArguments(
        output_dir=output_dir,
        run_name=wandb_run_name,

        # Training schedule
        num_train_epochs=epochs,
        max_steps=max_steps,

        # Batch size
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        gradient_accumulation_steps=gradient_accumulation_steps,

        # Optimization
        learning_rate=optimizer_config["learning_rate"],
        lr_scheduler_type=optimizer_config["lr_scheduler_type"],
        warmup_steps=optimizer_config["warmup_steps"],
        weight_decay=optimizer_config.get("weight_decay", 0.01),
        optim="adamw_torch",

        # Precision
        bf16=True,

        # Evaluation & logging
        eval_strategy="steps",
        eval_steps=eval_steps,
        logging_steps=logging_steps,
        logging_first_step=True,

        # Checkpointing
        save_strategy="steps",
        save_steps=save_steps,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,

        # Memory optimization
        gradient_checkpointing=True,

        # W&B
        report_to="wandb",

        # VL-specific
        remove_unused_columns=False,
    )

    # 3. Initialize callbacks
    lora_metrics_callback = WandBLoRAMetricsCallback(compute_freq=logging_steps)
    early_stopping = EarlyStoppingCallback(
        early_stopping_patience=early_stopping_patience,
        early_stopping_threshold=early_stopping_threshold
    )

    # 4. Create data collator
    data_collator = VLDataCollator(processor)

    # 5. Create trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=data_collator,
        callbacks=[lora_metrics_callback, early_stopping],
    )
    print("Starting training...")

    trainer.train()
    print("\nSaving model...")
    trainer.save_model(output_dir)
    test_dataset.save_to_disk(os.path.join(output_dir, "test_dataset"))

    wandb.run.summary["final_train_loss"] = trainer.state.log_history[-1].get("loss", None)
    wandb.run.summary["best_val_loss"] = lora_metrics_callback.best_val_loss

    artifact = wandb.Artifact(
        name=f"lora-checkpoint-{wandb.run.name}",
        type="model",
        description=f"LoRA adapter checkpoint",
        metadata={
            "best_val_loss": lora_metrics_callback.best_val_loss,
            "train_size": len(train_dataset),
            "val_size": len(val_dataset),
        }
    )

    artifact.add_dir(output_dir)
    wandb.log_artifact(artifact)

    print(f"✓ Artifact uploaded: {artifact.name}")

    # 10. Print summary
    print(f"\n{'='*70}")
    print("Training Complete!")
    print(f"Local model: {output_dir}")
    print(f"W&B Artifact: {artifact.name}")
    print(f"Dashboard: {wandb.run.url}")
    wandb.finish()
    return trainer

Load artifacts

In [34]:
def download_lora_from_wandb(
    artifact_name,
    base_model_name,
    wandb_entity=None,
    merge=False ):
    """
    Load a LoRA checkpoint from W&B Artifacts

    Args:
        artifact_name: Name of the artifact (e.g., "lora-checkpoint-quick-test-r4:latest")
        base_model_name: Base model to load
        wandb_entity: Your W&B username/team
        merge: If True, merge LoRA into base model for faster inference

    Returns:
        model: The model with LoRA adapter loaded (or merged)
    """
    api = wandb.Api()

    if wandb_entity:
        artifact_path = f"{wandb_entity}/{artifact_name}"
    else:
        artifact_path = artifact_name

    artifact = api.artifact(artifact_path)
    artifact_dir = artifact.download()

    print(f"✓ Downloaded to: {artifact_dir}")

In [35]:
def load_lora(base_model_name, artifact_dir, merge):

  """Load base model"""
  print(f"\n Loading base model: {base_model_name}")

  base_model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True
    )

  # Load LoRA adapter
  print(f"\n🔧 Loading LoRA adapter...")
  model = PeftModel.from_pretrained(base_model, artifact_dir)

  print(f"✓ LoRA adapter loaded!")
  print(f"  Metadata: {artifact.metadata}")

  # Optionally merge
  if merge:
      print(f"\n🔀 Merging LoRA into base model...")
      model = model.merge_and_unload()
      print(f"✓ Merged! Model is now standalone.")

  print(f"\n{'='*70}")
  print("✅ Model ready for inference!")
  print(f"{'='*70}\n")

  return model

Training Loop :)

In [ ]:
model_name = "Qwen/Qwen2.5-VL-3B-Instruct"
min_pixels = 256*28*28
max_pixels = 512*28*28
lora_r = 8
lora_alpha = 2*lora_r
lora_dropout = 0.05
target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]

TRAIN_SIZE =500
VAL_SIZE = 50
TEST_SIZE = 50

LEARNING_RATE = 5e-5
EPOCHS = 3

if __name__ == "__main__":
    """
    import wandb
    wandb.login()
    """
    print("="*70)
    print("STEP 1: Setting up model and processor")
    print("="*70)

    processor = AutoProcessor.from_pretrained(
        model_name,
        min_pixels=min_pixels,
        max_pixels=max_pixels
    )

    lora_config = create_lora_config_vl(lora_r=lora_r, lora_alpha=lora_alpha, lora_dropout = lora_dropout, target_modules=target_modules)

    model, _, trainable_params, total_params = setup_vl_model_and_processor(
        model_name=model_name,
        lora_config=lora_config,
        min_pixels=min_pixels,
        max_pixels=max_pixels
    )

    print(f"\n✓ Model: {trainable_params:,} trainable ({100*trainable_params/total_params:.2f}%)")

    # ========================================================================
    # STEP 2: Prepare datasets
    # ========================================================================
    print("\n" + "="*70)
    print("STEP 2: Preparing datasets with streaming")
    print("="*70)

    # Use small sample sizes for POC to minimize disk usage
    train_dataset, val_dataset, test_dataset = prepare_vqav2_datasets(
        processor=processor,
        train_size=500,  # Small for POC - only downloads 20 images
        val_size=100,
        test_size=100,
        cache_dir=None,  # Use temp directory
    )

    # ========================================================================
    # STEP 3: Configure optimizer
    # ========================================================================
    print("\n" + "="*70)
    print("STEP 3: Configuring optimizer")
    print("="*70)

    optimizer_config = setup_optimizer_scheduler(
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="cosine",
        warmup_steps=5,
        weight_decay=0.01,
    )

    print(f"✓ LR: {optimizer_config['learning_rate']}")
    print(f"✓ Scheduler: {optimizer_config['lr_scheduler_type']}")
    print(f"✓ Warmup: {optimizer_config['warmup_steps']} steps")

    # ========================================================================
    # STEP 4: Run training
    # ========================================================================
    print("\n" + "="*70)
    print("STEP 4: Starting training")
    print("="*70)

    # Prepare W&B config (any metadata you want to track)
    wandb_config = {
        "model_name": model_name,
        "lora_r": lora_r,
        "lora_alpha": lora_alpha,
        "min_pixels": min_pixels,
        "max_pixels": max_pixels,
        "dataset": "VQAv2",
    }

    trainer = train_vl_lora_with_wandb(
        # Required
        model=model,
        train_dataset=train_dataset,
        val_dataset=val_dataset,
        test_dataset=test_dataset,
        processor=processor,

        # W&B
        wandb_project="lora-vqav2-poc",
        wandb_run_name="quick-trial",  # You control the name!
        wandb_entity=None,
        wandb_config=wandb_config,

        # Output
        output_dir="./lora-vqav2-output",

        # Training schedule
        max_steps=30,
        eval_steps=10,
        logging_steps=5,

        # Batch size
        batch_size=1,
        gradient_accumulation_steps=2,

        # Optimizer
        optimizer_config=optimizer_config,

        # Early stopping
        early_stopping_patience=2,
    )

    print("\n" + "="*70)
    print("🎉 Complete! Check W&B dashboard.")
    print("="*70)

STEP 1: Setting up model and processor


preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Saving the dataset (0/2 shards):   0%|          | 0/100 [00:00<?, ? examples/s]

wandb: Adding directory to artifact (lora-vqav2-output)... wandb: WARNING Tried to log to step 30 that is less than the current step 32. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.
Done. 1.6s


✓ Artifact uploaded: lora-checkpoint-quick-trial

Training Complete!
Local model: ./lora-vqav2-output
W&B Artifact: lora-checkpoint-quick-trial
Dashboard: https://wandb.ai/katerina-itopoulos-srh-berlin/lora-vqav2-poc/runs/uc3tc2o1


eval/loss,█▃▁
eval/runtime,▆█▁
eval/samples_per_second,▃▁█
eval/steps_per_second,▃▁█
lora/effective_rank_mean,█▅▃▂▁▁
lora/effective_rank_std,▁▄▆▇██
lora/frobenius_norm_mean,▁▄▆▇██
lora/rank_utilization,█▅▃▂▁▁
lora/stable_rank_mean,█▅▃▂▁▁
train/epoch,▁▂▃▃▄▆▆▇███
+4,...



🎉 Complete! Check W&B dashboard.


In [36]:
import itertools

In [37]:
MODEL_NAME = "Qwen/Qwen3-VL-8B-Instruct"
MIN_PIXELS = 128*32*32  # 131,072 pixels (Qwen3-VL uses 32x32 patches!)
MAX_PIXELS = 256*32*32  # 262,144 pixels

# Hyperparameter search space
LORA_RANKS = [32, 64]
LEARNING_RATES = [1e-4, 2e-4, 5e-4]

# Fixed LoRA settings
LORA_ALPHA_MULTIPLIER = 2  # alpha = 2 * rank
LORA_DROPOUT = 0.05
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]  # LLM attention only

TRAIN_SIZE = 5000
VAL_SIZE = 500
TEST_SIZE = 500

# Training configuration
EPOCHS = 2
MAX_STEPS = -1  # Use -1 to ignore, let epochs control
EVAL_STRATEGY = "epoch"  # Evaluate at end of each epoch
SAVE_STRATEGY = "epoch"  # Save at end of each epoch
LOGGING_STEPS = 100  # Log every 100 steps (for progress tracking)

# Batch configuration
BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 4  # Effective batch size = 4

# Early stopping
EARLY_STOPPING_PATIENCE = 3  # liStop if no improvement for 3 epochs
EARLY_STOPPING_THRESHOLD = 0.01

# Weights & Biases
WANDB_PROJECT = "lora-qwen3-hyperparam-search"
WANDB_ENTITY = None

In [38]:
def generate_validation_configs():
    """Generate all combinations of rank and LR"""
    for rank, lr in itertools.product(LORA_RANKS, LEARNING_RATES):
        yield {
            "lora_r": rank,
            "lora_alpha": rank * LORA_ALPHA_MULTIPLIER,
            "learning_rate": lr,
        }

def create_validation_run_name(rank, lr, run_id):
    """Create descriptive run name"""
    return f"val_r{rank}_lr{lr:.0e}_id{run_id}"

In [39]:
!gcloud config set project secure-gizmo-439313-b2

Updated property [core/project].


To take a quick anonymous survey, run:
  $ gcloud survey



In [40]:
def prepare_vqav2_datasets_preprocessed_ultra_lean(
    processor,
    train_size=1000,
    val_size=200,
    test_size=200,
    gcs_bucket="where_you_lora_matters_thesis",
    gcs_prefix="datasets/preprocessed_vqa",
    save_frequency=100  # Save checkpoint every N samples
):
    """
    Ultra RAM-efficient with smart shard-level resume
    If train is at 3000/5000, resumes from 3000
    """
    from datasets import load_dataset, Dataset, concatenate_datasets
    from qwen_vl_utils import process_vision_info
    from itertools import islice
    import gcsfs
    import gc

    # GCS paths
    gcs_train_path = f"gs://{gcs_bucket}/{gcs_prefix}/train_{train_size}"
    gcs_val_path = f"gs://{gcs_bucket}/{gcs_prefix}/val_{val_size}"
    gcs_test_path = f"gs://{gcs_bucket}/{gcs_prefix}/test_{test_size}"

    fs = gcsfs.GCSFileSystem()

    def check_existing_progress(gcs_path, expected_size):
        """Check how many samples already exist"""
        try:
            if fs.exists(gcs_path.replace("gs://", "")):
                existing = Dataset.load_from_disk(gcs_path)
                current_size = len(existing)
                print(f"    Found {current_size}/{expected_size} samples")
                return current_size, existing
            else:
                print(f"    No existing data")
                return 0, None
        except Exception as e:
            print(f"    Error checking: {e}")
            return 0, None

    def format_and_preprocess(example):
        """Preprocess single example"""
        if isinstance(example.get('answer'), list):
            answer = example['answer'][0] if example['answer'] else ""
        else:
            answer = example.get('answer', example.get('multiple_choice_answer', ''))

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": example['image']},
                    {"type": "text", "text": example['question']}
                ]
            },
            {
                "role": "assistant",
                "content": [{"type": "text", "text": str(answer)}]
            }
        ]

        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        image_inputs, video_inputs = process_vision_info(messages)

        inputs = processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        )

        result = {
            "input_ids": inputs['input_ids'][0].tolist(),
            "attention_mask": inputs['attention_mask'][0].tolist(),
        }

        if 'pixel_values' in inputs and inputs['pixel_values'] is not None:
            pv = inputs['pixel_values']
            if pv.dim() == 5:
                pv = pv[0]
            result['pixel_values'] = pv.tolist()

        if 'image_grid_thw' in inputs and inputs['image_grid_thw'] is not None:
            igt = inputs['image_grid_thw']
            if igt.dim() == 2:
                igt = igt[0]
            result['image_grid_thw'] = igt.tolist()

        return result

    def process_split_with_resume(stream, total_size, split_name, gcs_path):
        """
        Process split with resume capability
        Resumes from where it left off if partially complete
        """
        print(f"\n{'='*70}")
        print(f"Processing {split_name}: {total_size} samples")
        print(f"{'='*70}")

        # Check existing progress
        current_size, existing_dataset = check_existing_progress(gcs_path, total_size)

        # If already complete, return it
        if current_size >= total_size:
            print(f"  ✓ {split_name} already complete!")
            return existing_dataset

        # If partial, skip to where we left off
        if current_size > 0:
            print(f"  🔄 Resuming from sample {current_size + 1}")
            # Skip already processed samples in stream
            for _ in range(current_size):
                next(islice(stream, 1))

        # Process remaining samples
        remaining = total_size - current_size
        num_chunks = (remaining + save_frequency - 1) // save_frequency

        for chunk_idx in range(num_chunks):
            chunk_start = current_size + (chunk_idx * save_frequency)
            chunk_end = min(chunk_start + save_frequency, total_size)
            chunk_size = chunk_end - chunk_start

            print(f"\n  Chunk {chunk_idx + 1}/{num_chunks}: samples {chunk_start + 1}-{chunk_end}")

            # Process chunk
            chunk_examples = []
            for i in range(chunk_size):
                try:
                    example = next(islice(stream, 1))
                    processed = format_and_preprocess(example)
                    chunk_examples.append(processed)

                    if (i + 1) % 20 == 0:
                        print(f"    Processed {i + 1}/{chunk_size}...")
                except StopIteration:
                    print(f"    ⚠️  Stream ended at {i}")
                    break
                except Exception as e:
                    print(f"    ⚠️  Skipped: {e}")
                    continue

            if not chunk_examples:
                print(f"    ⚠️  No samples in chunk, stopping")
                break

            # Create chunk dataset
            chunk_dataset = Dataset.from_list(chunk_examples)

            # Append to existing or create new
            if existing_dataset is not None:
                combined = concatenate_datasets([existing_dataset, chunk_dataset])
                combined.save_to_disk(gcs_path)
                print(f"    ✓ Appended {len(chunk_dataset)} samples (total: {len(combined)})")

                # Update existing_dataset reference
                del existing_dataset
                existing_dataset = combined
                del combined
            else:
                chunk_dataset.save_to_disk(gcs_path)
                print(f"    ✓ Created with {len(chunk_dataset)} samples")
                existing_dataset = chunk_dataset

            # FREE RAM!
            del chunk_examples, chunk_dataset
            gc.collect()

        # Reload final version from GCS
        final = Dataset.load_from_disk(gcs_path)
        print(f"  ✓ Final {split_name}: {len(final)} samples")

        del existing_dataset
        gc.collect()

        return final

    # Check overall progress
    print("🔍 Checking preprocessing progress...")

    # Process each split (with resume)
    print("\n" + "="*70)
    print("TRAIN SPLIT")
    train_stream = load_dataset("lmms-lab/VQAv2", split="validation", streaming=True)
    train_dataset = process_split_with_resume(train_stream, train_size, "train", gcs_train_path)
    del train_stream
    gc.collect()

    print("\n" + "="*70)
    print("VAL SPLIT")
    val_stream = load_dataset("lmms-lab/VQAv2", split="testdev", streaming=True)
    val_dataset = process_split_with_resume(val_stream, val_size, "val", gcs_val_path)
    del val_stream
    gc.collect()

    print("\n" + "="*70)
    print("TEST SPLIT")
    test_stream = load_dataset("lmms-lab/VQAv2", split="testdev", streaming=True)
    test_stream = test_stream.skip(val_size)
    test_dataset = process_split_with_resume(test_stream, test_size, "test", gcs_test_path)
    del test_stream
    gc.collect()

    print("\n" + "="*70)
    print("✓ ALL DATASETS COMPLETE")
    print("="*70)

    return train_dataset, val_dataset, test_dataset

In [44]:
def build_preprocessed_vqav2_qwen3vl(
    processor,
    train_size=1000,
    val_size=200,
    test_size=200,
    gcs_bucket="where_you_lora_matters_thesis",
    gcs_prefix="datasets/preprocessed_vqa",
    shard_size=256,       # how many samples per Parquet shard
):
    """
    One-time preprocessing of VQAv2 for Qwen3-VL-8B.
    - Streams original HF dataset
    - Precomputes text + vision inputs
    - Writes sharded Parquet files to GCS

    Writes:
      gs://{gcs_bucket}/{gcs_prefix}/train/train_shard_XXXX.parquet
      gs://{gcs_bucket}/{gcs_prefix}/val/val_shard_XXXX.parquet
      gs://{gcs_bucket}/{gcs_prefix}/test/test_shard_XXXX.parquet
    """
    import gc
    import torch
    from datasets import load_dataset, Dataset
    import gcsfs
    from qwen_vl_utils import process_vision_info

    fs = gcsfs.GCSFileSystem()

    # -------------------------
    # Single-example preprocessing
    # -------------------------
    def format_and_preprocess(example):
        # --- Get answer ---
        if isinstance(example.get("answer"), list):
            answer = example["answer"][0] if example["answer"] else ""
        else:
            answer = example.get(
                "answer", example.get("multiple_choice_answer", "")
            )

        # --- Build chat messages for Qwen3-VL ---
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": example["image"]},
                    {"type": "text", "text": example["question"]},
                ],
            },
            {
                "role": "assistant",
                "content": [{"type": "text", "text": str(answer)}],
            },
        ]

        text = processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )

        image_inputs, video_inputs = process_vision_info(messages)

        inputs = processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        )

        result = {
            "input_ids": inputs["input_ids"][0].tolist(),
            "attention_mask": inputs["attention_mask"][0].tolist(),
        }

        # Make vision tensors smaller & Arrow-friendly
        if "pixel_values" in inputs and inputs["pixel_values"] is not None:
            pv = inputs["pixel_values"]
            if pv.dim() == 5:  # (B, F, C, H, W) → handle single frame
                pv = pv[0]
            pv = pv.to(torch.float16)
            # numpy() so HF stores as dense array, not nested Python lists
            result["pixel_values"] = pv.numpy()

        if "image_grid_thw" in inputs and inputs["image_grid_thw"] is not None:
            igt = inputs["image_grid_thw"]
            if igt.dim() == 2:
                igt = igt[0]
            result["image_grid_thw"] = igt.numpy()

        return result

    # -------------------------
    # Split-level processing
    # -------------------------
    def process_split_to_shards(stream, total_size, split_name):
        """
        stream: HF IterableDataset (streaming=True)
        total_size: number of samples to take
        split_name: "train" | "val" | "test"
        """
        # GCS paths
        split_dir = f"{gcs_prefix}/{split_name}"
        gcs_split_dir = f"gs://{gcs_bucket}/{split_dir}"        # for HF/Datasets
        fs_split_prefix = f"{gcs_bucket}/{split_dir}"           # for gcsfs (NO gs://)

        print(f"\n{'='*70}")
        print(f"Processing {split_name}: {total_size} samples")
        print(f"{'='*70}")

        # Create iterator from streaming dataset
        stream_iter = iter(stream)

        # ---- Find existing shards to resume ----
        existing = sorted(fs.glob(f"{fs_split_prefix}/{split_name}_shard_*.parquet"))
        num_done = 0
        next_shard_idx = 0

        if existing:
            print(f"  Found {len(existing)} existing shard(s)")
            # Assume all but last shard are full size
            if len(existing) > 1:
                num_done += (len(existing) - 1) * shard_size

            # Read last shard length only
            last_uri = "gs://" + existing[-1]   # add scheme back
            last_ds = Dataset.from_parquet(last_uri)
            num_done += len(last_ds)
            next_shard_idx = len(existing)
            print(f"  Already processed {num_done}/{total_size} samples")
        else:
            print("  No existing shards, starting from scratch")

        if num_done >= total_size:
            print(f"  ✓ {split_name} already complete!")
            return

        # ---- Skip already-processed items in the stream ----
        if num_done > 0:
            print(f"  🔄 Skipping {num_done} samples in stream...")
            for _ in range(num_done):
                try:
                    next(stream_iter)
                except StopIteration:
                    break

        remaining = total_size - num_done

        # ---- Process remaining items in shards ----
        while remaining > 0:
            current_shard_size = min(shard_size, remaining)
            print(
                f"\n  Shard {next_shard_idx}: "
                f"samples {num_done + 1}-{num_done + current_shard_size}"
            )

            chunk_examples = []
            for i in range(current_shard_size):
                try:
                    example = next(stream_iter)
                except StopIteration:
                    print("    ⚠️ Stream ended early")
                    break

                try:
                    processed = format_and_preprocess(example)
                    chunk_examples.append(processed)
                except Exception as e:
                    print(f"    ⚠️ Skipped example: {e}")
                    continue

                if (i + 1) % 20 == 0:
                    print(f"    Processed {i + 1}/{current_shard_size}...")

            if not chunk_examples:
                print("    ⚠️ No examples processed in shard, stopping")
                break

            shard_ds = Dataset.from_list(chunk_examples)
            shard_path = f"{gcs_split_dir}/{split_name}_shard_{next_shard_idx:04d}.parquet"
            print(f"    Saving shard to {shard_path}")
            shard_ds.to_parquet(shard_path)

            num_done += len(shard_ds)
            remaining = total_size - num_done
            next_shard_idx += 1

            del shard_ds, chunk_examples
            gc.collect()

        print(f"\n  ✓ Final {split_name}: {num_done}/{total_size} samples")

    # -------------------------
    # Run all splits
    # -------------------------
    print("🔍 Starting one-time preprocessing for Qwen3-VL...")

    # TRAIN
    train_stream = load_dataset("lmms-lab/VQAv2", split="validation", streaming=True)
    process_split_to_shards(train_stream, train_size, "train")
    del train_stream
    gc.collect()

    # VAL
    val_stream = load_dataset("lmms-lab/VQAv2", split="testdev", streaming=True)
    process_split_to_shards(val_stream, val_size, "val")
    del val_stream
    gc.collect()

    # TEST
    test_stream = load_dataset("lmms-lab/VQAv2", split="testdev", streaming=True)
    test_stream = test_stream.skip(val_size)
    process_split_to_shards(test_stream, test_size, "test")
    del test_stream
    gc.collect()

    print("\n" + "=" * 70)
    print("✓ ALL PREPROCESSED SHARDS WRITTEN TO GCS")
    print("=" * 70)

In [ ]:
"""
Preprocess VQAv2 datasets and save to GCS
Run this ONCE before validation experiments
"""

# Configuration
MODEL_NAME = "Qwen/Qwen3-VL-8B-Instruct"
MIN_PIXELS = 128*32*32
MAX_PIXELS = 256*32*32

# Dataset sizes to preprocess
TRAIN_SIZE = 20000
VAL_SIZE = 2000
TEST_SIZE = 5000

GCS_BUCKET = "where_you_lora_matters_thesis"
GCS_PREFIX = "datasets/preprocessed_vqa_6k"

if __name__ == "__main__":
    print("="*70)
    print("PREPROCESSING VQAv2 DATASETS FOR THESIS EXPERIMENTS")
    print("="*70)
    print(f"Train size: {TRAIN_SIZE}")
    print(f"Val size: {VAL_SIZE}")
    print(f"Test size: {TEST_SIZE}")
    print(f"GCS bucket: gs://{GCS_BUCKET}/{GCS_PREFIX}/")
    print("="*70)

    # Load processor
    print("\n📦 Loading processor...")
    processor = AutoProcessor.from_pretrained(
        MODEL_NAME,
        min_pixels=MIN_PIXELS,
        max_pixels=MAX_PIXELS,
        trust_remote_code=True
    )

    # Preprocess and save to GCS
    print("\n📊 Preprocessing datasets...")
    build_preprocessed_vqav2_qwen3vl(
      processor=processor,
      train_size=TRAIN_SIZE,
      val_size=VAL_SIZE,
      test_size=TEST_SIZE,
      gcs_bucket=GCS_BUCKET,
      gcs_prefix=GCS_PREFIX,
      shard_size=256
  )

    print("\n" + "="*70)
    print("✓ PREPROCESSING COMPLETE")
    print("="*70)

In [ ]:
if __name__ == "__main__":
    import wandb
    wandb.login()

    validation_configs = list(generate_validation_configs())
    total_runs = len(validation_configs)

    print("="*70)
    print(f"HYPERPARAMETER VALIDATION: {total_runs} configurations")
    print("="*70)
    print(f"Strategy: LLM-only LoRA (attention layers)")
    print(f"Ranks: {LORA_RANKS}")
    print(f"Learning rates: {LEARNING_RATES}")
    print(f"Training samples: {TRAIN_SIZE}")
    print(f"Epochs: {EPOCHS}")
    print("="*70)

    print("\n" + "="*70)
    print("LOADING PREPROCESSED DATASETS FROM GCS")
    print("="*70)

    # Load processor (just for collator, not for preprocessing)
    processor = AutoProcessor.from_pretrained(
        MODEL_NAME,
        min_pixels=MIN_PIXELS,
        max_pixels=MAX_PIXELS,
        trust_remote_code=True
    )

    # Load preprocessed datasets from GCS (instant!)
    from datasets import Dataset

    GCS_BUCKET = "where_you_lora_matters_thesis"
    GCS_PREFIX = "datasets/preprocessed_vqa"

    print("Loading train dataset...")
    train_dataset = Dataset.load_from_disk(
        f"gs://{GCS_BUCKET}/{GCS_PREFIX}/train_{TRAIN_SIZE}"
    )

    print("Loading val dataset...")
    val_dataset = Dataset.load_from_disk(
        f"gs://{GCS_BUCKET}/{GCS_PREFIX}/val_{VAL_SIZE}"
    )

    print("Loading test dataset...")
    test_dataset = Dataset.load_from_disk(
        f"gs://{GCS_BUCKET}/{GCS_PREFIX}/test_{TEST_SIZE}"
    )

    print(f"✓ Train: {len(train_dataset)} samples")
    print(f"✓ Val: {len(val_dataset)} samples")
    print(f"✓ Test: {len(test_dataset)} samples")

    results = []

    for run_id, config in enumerate(validation_configs, 1):
        rank = config["lora_r"]
        lr = config["learning_rate"]
        alpha = config["lora_alpha"]

        print("\n" + "="*70)
        print(f"VALIDATION RUN {run_id}/{total_runs}")
        print("="*70)
        print(f"LoRA rank: {rank}")
        print(f"LoRA alpha: {alpha} ({LORA_ALPHA_MULTIPLIER}x)")
        print(f"Learning rate: {lr}")
        print(f"Target modules: {TARGET_MODULES}")

        try:
            lora_config = create_lora_config_vl(
                lora_r=rank,
                lora_alpha=alpha,
                lora_dropout=LORA_DROPOUT,
                target_modules=TARGET_MODULES
            )

            model, _, trainable_params, total_params = setup_vl_model_and_processor(
                model_name=MODEL_NAME,
                lora_config=lora_config,
                min_pixels=MIN_PIXELS,
                max_pixels=MAX_PIXELS
            )

            print(f"✓ Trainable params: {trainable_params:,} ({100*trainable_params/total_params:.2f}%)")

            optimizer_config = setup_optimizer_scheduler(
                learning_rate=lr,
                lr_scheduler_type="cosine",
                warmup_steps=10,
                weight_decay=0.01,
            )

            wandb_config = {
                "experiment_type": "hyperparameter_validation",
                "placement_strategy": "llm_only",
                "model_name": MODEL_NAME,
                "lora_r": rank,
                "lora_alpha": alpha,
                "lora_alpha_multiplier": LORA_ALPHA_MULTIPLIER,
                "lora_dropout": LORA_DROPOUT,
                "target_modules": TARGET_MODULES,
                "num_target_modules": len(TARGET_MODULES),
                "learning_rate": lr,
                "lr_scheduler_type": "cosine",
                "min_pixels": MIN_PIXELS,
                "max_pixels": MAX_PIXELS,
                "train_size": TRAIN_SIZE,
                "val_size": VAL_SIZE,
                "test_size": TEST_SIZE,
                "epochs": EPOCHS,
                "trainable_params": trainable_params,
                "trainable_percentage": 100*trainable_params/total_params,
            }

            run_name = create_validation_run_name(rank, lr, run_id)
            output_dir = f"./validation_outputs/{run_name}"

            print(f"\nStarting training: {run_name}")

            trainer = train_vl_lora_with_wandb(
                # Required
                model=model,
                train_dataset=train_dataset,
                val_dataset=val_dataset,
                test_dataset=test_dataset,
                processor=processor,

                # W&B
                wandb_project=WANDB_PROJECT,
                wandb_run_name=run_name,
                wandb_entity=WANDB_ENTITY,
                wandb_config=wandb_config,

                # Output
                output_dir=output_dir,

                # Training schedule
                epochs=EPOCHS,
                max_steps=-1,
                eval_steps=None,  # Epoch-based eval
                logging_steps=LOGGING_STEPS,
                save_steps=None,  # Epoch-based save

                # Batch size
                batch_size=BATCH_SIZE,
                gradient_accumulation_steps=GRAD_ACCUM_STEPS,

                # Optimizer
                optimizer_config=optimizer_config,

                # Early stopping
                early_stopping_patience=EARLY_STOPPING_PATIENCE,
            )

            # Get best validation loss
            best_val_loss = min([log.get("eval_loss", float('inf'))
                                for log in trainer.state.log_history
                                if "eval_loss" in log])

            # Log success
            results.append({
                "run_id": run_id,
                "run_name": run_name,
                "status": "success",
                "lora_r": rank,
                "learning_rate": lr,
                "best_val_loss": best_val_loss,
                "trainable_params": trainable_params,
            })

            print(f"✓ Run {run_id} completed successfully")
            print(f"✓ Best val loss: {best_val_loss:.4f}")

            # Clean up
            del model
            del trainer
            torch.cuda.empty_cache()

        except Exception as e:
            print(f"✗ Run {run_id} failed with error: {str(e)}")
            import traceback
            traceback.print_exc()

            results.append({
                "run_id": run_id,
                "run_name": create_validation_run_name(rank, lr, run_id),
                "status": "failed",
                "lora_r": rank,
                "learning_rate": lr,
                "error": str(e),
            })

            # Clean up even on failure
            if 'model' in locals():
                del model
            if 'trainer' in locals():
                del trainer
            torch.cuda.empty_cache()

            continue

    print("\n" + "="*70)
    print("VALIDATION COMPLETE - RESULTS ANALYSIS")
    print("="*70)

    successful_runs = [r for r in results if r["status"] == "success"]

    if successful_runs:
        # Sort by best validation loss
        successful_runs.sort(key=lambda x: x["best_val_loss"])

        print("\nTop 3 configurations:")
        print("-" * 70)
        for i, result in enumerate(successful_runs[:3], 1):
            print(f"{i}. Rank={result['lora_r']}, LR={result['learning_rate']:.0e}")
            print(f"   Val Loss: {result['best_val_loss']:.4f}")
            print(f"   Trainable params: {result['trainable_params']:,}")
            print()

        best_config = successful_runs[0]
        print("="*70)
        print("RECOMMENDED CONFIGURATION FOR MAIN EXPERIMENTS:")
        print("="*70)
        print(f"LoRA Rank: {best_config['lora_r']}")
        print(f"LoRA Alpha: {best_config['lora_r'] * LORA_ALPHA_MULTIPLIER}")
        print(f"Learning Rate: {best_config['learning_rate']:.0e}")
        print(f"Best Val Loss: {best_config['best_val_loss']:.4f}")
        print("="*70)

        recommendation = {
            "lora_r": best_config['lora_r'],
            "lora_alpha": best_config['lora_r'] * LORA_ALPHA_MULTIPLIER,
            "learning_rate": best_config['learning_rate'],
            "best_val_loss": best_config['best_val_loss'],
            "lora_dropout": LORA_DROPOUT,
            "validation_date": datetime.now().isoformat(),
        }

        with open("recommended_hyperparameters.json", "w") as f:
            json.dump(recommendation, f, indent=2)

        print(f"\n✓ Saved to: recommended_hyperparameters.json")

    else:
        print("\n✗ No successful runs to analyze")

    # Report failures
    failed_runs = [r for r in results if r["status"] == "failed"]
    if failed_runs:
        print("\nFailed configurations:")
        for result in failed_runs:
            print(f"  - Rank={result['lora_r']}, LR={result['learning_rate']:.0e}")
            print(f"    Error: {result.get('error', 'Unknown')}")

    print(f"\n✓ Check W&B dashboard: wandb.ai")
    print(f"✓ Total runs: {len(results)} ({len(successful_runs)} successful)")
    print("="*70)

HYPERPARAMETER VALIDATION: 6 configurations
Strategy: LLM-only LoRA (attention layers)
Ranks: [32, 64]
Learning rates: [0.0001, 0.0002, 0.0005]
Training samples: 1000
Epochs: 2

PREPARING DATASETS (once for all validation runs)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Resolving data files:   0%|          | 0/143 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/68 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/36 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/143 [00:00<?, ?it/s]

✓ Downloaded 1400 samples
Preprocessing train set...
